In [ ]:
import os
import glob
import json
from collections import defaultdict
import numpy as np
import matplotlib as plt

## Data analysis
In this document I will go further in depth on my data quality.

In [5]:
def load_yolo_labels(label_dir):
    data = []

    label_files = glob.glob(os.path.join(label_dir, '*.txt'))

    for file in label_files:
        image_id = os.path.basename(file).split('.')[0]

        with open(file, 'r') as f:
            lines = f.readlines()

        for line in lines:
            parts = list(map(float, line.strip().split()))
            if len(parts) != 5:
                continue # Skip lines that don't have exactly 5 parts, since yolo labels should have 5 parts: class_id, x_center, y_center, width, height

            cls, x, y, w, h = parts

            data.append({
                "image_id": image_id,
                "class": int(cls),
                "x": x,
                "y": y,
                "w": w,
                "h": h,
                "area": w * h
            })

    return data

In [6]:
def group_by_image(data):
    grouped = defaultdict(list)
    
    for data_point in data:
        grouped[data_point['image_id']].append(data_point)

    return dict(grouped)

grouped_labels = group_by_image(load_yolo_labels(r"S3-celetial-object-detection\Data\DSYD raw data\DeepSpaceYoloDataset\labels"))

with open('grouped_labels.json', 'w') as f:
    json.dump(grouped_labels, f, indent=4)

In [ ]:
def data_stats(data):
    widths = [d['w'] for d in data]
    heights = [d['h'] for d in data]
    areas = [d['area'] for d in data]
    
    images = [d['image_id'] for d in data]
    objects_per_image = defaultdict(int)

    for im in images:
        objects_per_image[im] += 1

    print("Dataset statistics:")
    print(f'Total objects: {len(data)}')
    print(f'Total images with objects: {len(objects_per_image)}')

    print('\nBounding box stats:')
    print(f"Width  - mean: {np.mean(widths):.4f}, min: {np.min(widths):.4f}, max: {np.max(widths):.4f}")
    print(f"Height - mean: {np.mean(heights):.4f}, min: {np.min(heights):.4f}, max: {np.max(heights):.4f}")
    print(f"Area   - mean: {np.mean(areas):.4f}, min: {np.min(areas):.6f}, max: {np.max(areas):.4f}")

    print("\nObjects per image:")
    print(f"Mean: {np.mean(list(objects_per_image.values())):.2f}")
    print(f"Max: {np.max(list(objects_per_image.values()))}") 

In [ ]:
def plot_distributions(data):
    widths = [d['w'] for d in data]
    heights = [d['h'] for d in data]
    areas = [d['area'] for d in data]

    plt.figure()
    plt.hist(widths, bins=50)
    plt.title("Bounding Box Width Distributions")
    plt.show()

    plt.figure()
    plt.hist(heights, bins=50)
    plt.title("Bounding Box Height Distributions")
    plt.show()

    plt.figure()
    plt.hist(areas, bins=50)
    plt.title("Bounding Box Area Distributions")
    plt.show()